# BOUNDING BOXES TEST

In [ ]:
import sys
import asyncio
import os
from src.knowledge_base.service import DocumentService
async def main():
    service = DocumentService()
    query = "what is the technical data for the materials?"
    
    # Needs to initialize settings, VectorStore, etc.
    from src.core.config import get_settings
    settings = get_settings()
    
    print(f"Searching for: {query}")
    try:
        chunks = await service.search(query=query, top_k=3)
        for i, c in enumerate(chunks):
            print(f"\n--- Chunk {i+1} ---")
            print(f"File: {c.get('filename')}")
            print(f"BBoxes: {c.get('bboxes')}")
            print(f"Section: {c.get('section_path')}")
            print(f"Text snippet: {c.get('content')[:200]}...")
    except Exception as e:
        import traceback
        traceback.print_exc()
if __name__ == "__main__":
    asyncio.run(main())

# API TESTING

In [ ]:
import urllib.request
import json
import urllib.parse

def main():
    url = "http://localhost:8001/api/search"
    query = "what is the technical data for the materials?"
    data = json.dumps({"query": query}).encode("utf-8")
    req = urllib.request.Request(url, data=data, headers={'Content-Type': 'application/json'})
    
    try:
        response = urllib.request.urlopen(req)
        result = json.loads(response.read().decode("utf-8"))
        
        print("Search successful.")
        print("Search successful, checking first citation.")
        for i, c in enumerate(result.get("results", [])):
            print(f"\n--- Citation {i+1} ---")
            bbox = c.get('bbox', [])
            formatted_bbox = [round(x, 2) for x in bbox] if bbox else []
            print(f"File: {c.get('filename')}")
            print(f"BBox: {formatted_bbox}")
            print(f"Location: page {c.get('page_no')}")
            print(f"Text snippet: {c.get('content')[:100]}...")
            
    except Exception as e:
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()



# TESTING AND BENCHMARKING LATENCY IN API CALLS

In [ ]:
import os
import time
import statistics
from openai import OpenAI, AzureOpenAI

# -------------------------
# Configuration
# -------------------------

MODEL_NAME = "text-embedding-3-large"  # OpenAI model
AZURE_DEPLOYMENT = "your-azure-embedding-deployment-name"
TEST_TEXT = "This is a test sentence to benchmark embedding latency." * 20
RUNS = 10

# -------------------------
# Clients
# -------------------------

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
)

azure_client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)

# -------------------------
# Benchmark Function
# -------------------------

def benchmark_openai():
    times = []
    for _ in range(RUNS):
        start = time.perf_counter()
        openai_client.embeddings.create(
            model=MODEL_NAME,
            input=TEST_TEXT,
        )
        end = time.perf_counter()
        times.append(end - start)
    return times


def benchmark_azure():
    times = []
    for _ in range(RUNS):
        start = time.perf_counter()
        azure_client.embeddings.create(
            model=AZURE_DEPLOYMENT,  # Azure uses deployment name
            input=TEST_TEXT,
        )
        end = time.perf_counter()
        times.append(end - start)
    return times


# -------------------------
# Run Benchmarks
# -------------------------

print("Running OpenAI benchmark...")
openai_times = benchmark_openai()

print("Running Azure benchmark...")
azure_times = benchmark_azure()

# -------------------------
# Results
# -------------------------

def summarize(name, times):
    print(f"\n{name} Results:")
    print(f"  Avg:  {statistics.mean(times):.4f} sec")
    print(f"  Min:  {min(times):.4f} sec")
    print(f"  Max:  {max(times):.4f} sec")
    print(f"  Std:  {statistics.stdev(times):.4f} sec")

summarize("OpenAI", openai_times)
summarize("Azure", azure_times)

In [ ]:
import sys
import os

# Add src to path
sys.path.append(os.path.join(os.getcwd(), 'src'))

try:
    from src.agent.agent import agent
    print("Successfully imported agent")
    tools = getattr(agent, "_function_tools", None) or getattr(agent, "tools", None)
    print([attr for attr in dir(agent) if "tool" in attr.lower()])
    print(f"Agent tools: {[getattr(t, 'name', str(t)) for t in tools] if tools else 'No public tool list found'}")
except Exception as e:
    import traceback
    traceback.print_exc()
    sys.exit(1)